# Notebook 2 — Treinamento: Fine-Tuning Eficiente com LoRA

## Objetivos

1. Preparar dados tokenizados para treino com Transformer
2. Configurar adaptadores LoRA (Hu et al., 2022) — Snippet 2 do material
3. Treinar modelo com Hugging Face Trainer (PEFT)
4. Comparar eficiência: Full Fine-Tuning vs LoRA
5. Salvar adaptadores treinados

## Conexão Teórica

> *"LoRA injeta uma atualização de baixo rank W = W₀ + BA, onde
> B ∈ ℝ^(d×r) e A ∈ ℝ^(r×k) são matrizes treináveis de rank r ≪ min(d, k).
> Na prática, valores de r entre 4 e 64 demonstram desempenho competitivo
> com fine-tuning completo em diversas tarefas."*
> — Material da Aula 5, seção 'Low-Rank Adaptation (LoRA)' (Hu et al., 2022)

> *"A inicialização: A ~ N(0, σ²), B = 0, garante ΔW = BA = 0 no início,
> preservando conhecimento do pré-treinamento."*
> — Material da Aula 5, seção LoRA

In [ ]:
# Imports
import sys
import os
import warnings

sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.data_preprocessing import DataPreprocessor
from src.model import PEFTModelWrapper
from src.training import create_datasets, train_with_peft, compare_training_efficiency
from src.evaluation import plot_parameter_comparison, plot_training_curves
from src.utils import count_parameters, save_model, save_metrics, set_seed, timer

set_seed(42)
sns.set_theme(style="whitegrid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1. Carregamento e Preparação dos Dados

Utilizamos o `DataPreprocessor` para carregar o dataset gerado
no Notebook 01 e preparar splits treino/validação/teste.

In [ ]:
DATASET_PATH = os.path.join("..", "data", "raw", "dataset.csv")

preprocessor = DataPreprocessor(max_length=128, test_size=0.2, val_size=0.1, random_state=42)
preprocessor.load_data(DATASET_PATH)
preprocessor.clean_data()
preprocessor.prepare_features()
train_df, val_df, test_df = preprocessor.split_data()

print(f"Classes: {preprocessor.label_map}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 2. Carregamento do Modelo Base

Carregamos DistilBERT como modelo base. Em cenários com GPU,
poderíamos ativar quantização 4-bit (QLoRA, Dettmers et al., 2023)
via `quantize_4bit=True`.

In [ ]:
# Carregar modelo base
wrapper = PEFTModelWrapper(
    model_name="distilbert-base-uncased",
    num_labels=preprocessor.num_labels,
    device=DEVICE,
)
base_model = wrapper.load_base_model(quantize_4bit=False)

# Contagem de parâmetros (full fine-tuning: todos treináveis)
base_stats = count_parameters(base_model)
print(f"Modelo base: {base_stats['total']:,} parâmetros (100% treináveis)")

## 3. Configuração LoRA

Aplicamos adaptadores LoRA conforme **Snippet 2 do material**:

```python
# Material da Aula 5, Snippet 2
lora_config = LoraConfig(
    r=16,                    # Rank dos adaptadores
    lora_alpha=32,           # Fator de escalonamento
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
```

Para DistilBERT (classificação), adaptamos: `target_modules=["q_lin", "v_lin"]`
e `task_type="SEQ_CLS"`.

O escalonamento α/r = 32/16 = 2 modula a magnitude das atualizações.

In [ ]:
# Aplicar LoRA (Hu et al., 2022)
peft_model = wrapper.apply_lora(
    r=16,                          # Rank — dimensão intrínseca
    lora_alpha=32,                 # Escalonamento: α/r = 32/16 = 2
    target_modules=["q_lin", "v_lin"],  # Módulos de atenção do DistilBERT
    lora_dropout=0.05,             # Regularização
    task_type="SEQ_CLS",           # Classificação de sequência
)

# Comparação de parâmetros (Tabela 2 do material)
peft_stats = count_parameters(peft_model)
reduction = (1 - peft_stats['trainable'] / base_stats['total']) * 100
print(f"\nLoRA aplicado:")
print(f"  Treináveis: {peft_stats['trainable']:,} ({peft_stats['trainable_pct']:.4f}%)")
print(f"  Congelados: {peft_stats['frozen']:,}")
print(f"  Redução: {reduction:.2f}% menos parâmetros a treinar")

In [ ]:
# Visualizar parâmetros LoRA adicionados
print("Parâmetros LoRA (treináveis):")
print("=" * 60)
for name, param in peft_model.named_parameters():
    if param.requires_grad:
        print(f"  {name:50s} {str(list(param.shape)):>15s}  ({param.numel():,})")

## 4. Criação dos Datasets para o Trainer

Convertemos os DataFrames em `TextClassificationDataset` compatíveis
com o Hugging Face Trainer.

In [ ]:
# Criar datasets PyTorch
train_dataset, val_dataset, test_dataset = create_datasets(
    train_df, val_df, test_df,
    tokenizer=wrapper.tokenizer,
    max_length=128,
    text_col="text",
    label_col="label_id",
)

print(f"Train dataset: {len(train_dataset)} amostras")
print(f"Val dataset:   {len(val_dataset)} amostras")
print(f"Test dataset:  {len(test_dataset)} amostras")

# Verificar formato de um exemplo
sample = train_dataset[0]
print(f"\nFormato de exemplo:")
for k, v in sample.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

## 5. Treinamento com LoRA

Treinamos usando `train_with_peft()`, que encapsula o Hugging Face
Trainer com configurações otimizadas para PEFT:

- **Learning rate**: 2e-4 (recomendado para LoRA, Hu et al., 2022, Seção 5)
- **Warmup**: 10% dos passos (estabilidade numérica)
- **Weight decay**: 0.01 (regularização leve)

> **Nota**: Em CPU, o treinamento é mais lento. Com GPU, ative `fp16=True`
> para mixed precision (Dettmers et al., 2023).

In [ ]:
# Diretório de saída
OUTPUT_DIR = os.path.join("..", "outputs", "models", "lora_adapter")

# Treinar com LoRA
# Nota: num_epochs=3 para demonstração; em produção use mais épocas
with timer("Treinamento LoRA") as t:
    train_results = train_with_peft(
        model=peft_model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        output_dir=OUTPUT_DIR,
        num_epochs=3,
        batch_size=16,
        learning_rate=2e-4,     # Recomendado por Hu et al. (2022)
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=25,
        fp16=torch.cuda.is_available(),
    )

print(f"\nResultados do Treinamento:")
print(f"  Loss final: {train_results['train_loss']:.4f}")
print(f"  Val Accuracy: {train_results['eval_accuracy']:.4f}")
print(f"  Val F1 (macro): {train_results['eval_f1_macro']:.4f}")
print(f"  Tempo: {t['elapsed']:.1f}s")

## 6. Visualização das Curvas de Treinamento

In [ ]:
# Curvas de treinamento simples
# (Em treinamento real, extrairíamos do Trainer.state.log_history)
fig = plot_training_curves(
    train_losses=[train_results["train_loss"]],  # Resumo por época
    val_accuracies=[train_results["eval_accuracy"]],
    title="Curvas de Treinamento — Fine-Tuning com LoRA (r=16)\n(Hu et al., 2022)",
)
plt.show()

## 7. Comparação de Eficiência

Comparamos parâmetros e memória entre Full Fine-Tuning e LoRA,
reproduzindo a análise da Tabela 2 do material.

In [ ]:
# Comparação de eficiência
comparison_df = compare_training_efficiency(base_model, peft_model)

print("Comparação de Eficiência (Tabela 2 do material):")
print("=" * 80)
print(comparison_df.to_string(index=False))

fig = plot_parameter_comparison(
    comparison_df,
    title="Full Fine-Tuning vs LoRA — Eficiência de Parâmetros",
    save_path=os.path.join("..", "outputs", "figures", "parameter_comparison.png"),
)
plt.show()

## 8. Salvamento do Modelo

Salvamos apenas os adaptadores LoRA (poucos MBs), não o modelo
base inteiro. Na inferência, basta recarregar o modelo base e
aplicar os adaptadores — ou fundir pesos (W = W₀ + BA).

In [ ]:
# Salvar adaptadores PEFT (apenas os pesos LoRA)
ADAPTER_PATH = os.path.join("..", "outputs", "models", "lora_adapter_final")
save_model(peft_model, ADAPTER_PATH, save_peft_only=True)

# Salvar métricas de treinamento
METRICS_PATH = os.path.join("..", "outputs", "logs", "train_metrics.json")
save_metrics(train_results, METRICS_PATH)

# Salvar label_map para uso na avaliação
import json
LABEL_MAP_PATH = os.path.join("..", "outputs", "logs", "label_map.json")
os.makedirs(os.path.dirname(LABEL_MAP_PATH), exist_ok=True)
with open(LABEL_MAP_PATH, "w") as f:
    json.dump(preprocessor.label_map, f)

print(f"Adaptadores salvos em: {ADAPTER_PATH}")
print(f"Métricas salvas em: {METRICS_PATH}")

# Verificar tamanho dos adaptadores vs modelo completo
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
) / (1024 ** 2)
print(f"\nTamanho dos adaptadores: {adapter_size:.2f} MB")
print(f"Tamanho do modelo completo: ~{count_parameters(base_model)['total'] * 4 / 1024**2:.0f} MB (float32)")

## Resumo

Neste notebook, implementamos:

1. **LoRA (r=16, α=32)**: Adaptadores de baixo rank em q_lin e v_lin
2. **Treinamento eficiente**: Hugging Face Trainer com lr=2e-4
3. **Economia de parâmetros**: ~99.8% dos pesos congelados
4. **Armazenamento**: Apenas adaptadores salvos (poucos MBs)

### Próximo Notebook

No **Notebook 03 (Avaliação)**, avaliaremos o modelo treinado,
realizaremos benchmark de latência e exportaremos para ONNX.